In [1]:
# ================================================================
# REAL INSTAGRAM DATA - ENGAGEMENT TARGET DISCOVERY
# AI-Based Instagram Engagement Prediction and Content Optimization
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")

# ================================================================
# PROJECT PATHS
# ================================================================

PROJECT_ROOT = Path(r"d:\newwwwwwww\AiBasedInstagramPrediction")

DATASETS_DIR = PROJECT_ROOT / "datasets"
RAW_DIR = DATASETS_DIR / "raw"
RESULTS_DIR = PROJECT_ROOT / "results" / "real_multimodal_evaluation"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("REAL INSTAGRAM ENGAGEMENT TARGET DISCOVERY")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DIR)

# ================================================================
# SEARCH ALL CSV FILES
# ================================================================

csv_files = list(RAW_DIR.rglob("*.csv"))

print("\n" + "=" * 70)
print("CSV FILES FOUND")
print("=" * 70)

if not csv_files:
    raise FileNotFoundError("No CSV files were found in the raw dataset directory.")

for i, file in enumerate(csv_files, 1):
    print(f"{i:02d}. {file}")

# ================================================================
# LOAD ALL CSV FILES
# ================================================================

datasets = {}

print("\n" + "=" * 70)
print("LOADING DATASETS")
print("=" * 70)

for file in csv_files:

    try:
        df = pd.read_csv(file, low_memory=False)

        key = file.stem

        # Avoid duplicate keys
        original_key = key
        counter = 2

        while key in datasets:
            key = f"{original_key}_{counter}"
            counter += 1

        datasets[key] = {
            "path": file,
            "data": df
        }

        print(
            f"\n{key}"
            f"\n  Rows    : {len(df):,}"
            f"\n  Columns : {len(df.columns)}"
        )

        print("  Columns:")

        for col in df.columns:
            print(f"    - {col}")

    except Exception as e:

        print(f"\nFAILED: {file}")
        print(f"ERROR : {e}")

# ================================================================
# ENGAGEMENT / TARGET KEYWORDS
# ================================================================

TARGET_KEYWORDS = [
    "like",
    "likes",
    "comment",
    "comments",
    "share",
    "shares",
    "save",
    "saves",
    "engagement",
    "engagement_rate",
    "engagementrate",
    "reach",
    "impression",
    "impressions",
    "interaction",
    "interactions",
    "views",
    "view_count",
    "likes_count",
    "comments_count",
    "shares_count",
    "saves_count"
]

print("\n" + "=" * 70)
print("ENGAGEMENT / TARGET FIELD DISCOVERY")
print("=" * 70)

target_candidates = []

for name, info in datasets.items():

    df = info["data"]

    print(f"\n{'-' * 70}")
    print(f"DATASET: {name}")
    print(f"FILE   : {info['path']}")
    print(f"{'-' * 70}")

    found = []

    for col in df.columns:

        col_clean = str(col).lower().strip()
        normalized = re.sub(r"[^a-z0-9]", "", col_clean)

        matched_keywords = []

        for keyword in TARGET_KEYWORDS:

            keyword_normalized = re.sub(
                r"[^a-z0-9]",
                "",
                keyword.lower()
            )

            if keyword_normalized in normalized:
                matched_keywords.append(keyword)

        if matched_keywords:

            found.append(col)

            target_candidates.append({
                "dataset": name,
                "column": col,
                "dtype": str(df[col].dtype),
                "missing": int(df[col].isna().sum()),
                "missing_pct": float(df[col].isna().mean() * 100),
                "unique": int(df[col].nunique(dropna=True)),
                "matched_keywords": ", ".join(matched_keywords)
            })

    if found:

        print("\nPotential engagement fields:")

        for col in found:

            series = df[col]

            print(f"\n  COLUMN: {col}")
            print(f"    Data type : {series.dtype}")
            print(f"    Missing   : {series.isna().sum():,}")
            print(f"    Unique    : {series.nunique(dropna=True):,}")

            if pd.api.types.is_numeric_dtype(series):

                print(f"    Minimum   : {series.min()}")
                print(f"    Median    : {series.median()}")
                print(f"    Mean      : {series.mean()}")
                print(f"    Maximum   : {series.max()}")

            else:

                print("    Sample values:")

                values = series.dropna().astype(str).unique()[:10]

                for value in values:
                    print(f"      {value[:150]}")

    else:

        print("\n  No obvious engagement fields found.")

# ================================================================
# SAVE TARGET CANDIDATES
# ================================================================

if target_candidates:

    target_df = pd.DataFrame(target_candidates)

    target_file = RESULTS_DIR / "real_engagement_target_candidates.csv"

    target_df.to_csv(target_file, index=False)

    print("\n" + "=" * 70)
    print("TARGET CANDIDATES SUMMARY")
    print("=" * 70)

    print(target_df.to_string(index=False))

    print("\nSaved:")
    print(target_file)

else:

    print("\n" + "=" * 70)
    print("NO ENGAGEMENT TARGET CANDIDATES FOUND")
    print("=" * 70)

# ================================================================
# IDENTIFIER / IMAGE MATCHING FIELD DISCOVERY
# ================================================================

MATCH_KEYWORDS = [
    "image",
    "image_file",
    "imagefile",
    "filename",
    "file_name",
    "post_id",
    "media_id",
    "id",
    "url",
    "shortcode",
    "caption"
]

print("\n" + "=" * 70)
print("IMAGE / POST MATCHING FIELD DISCOVERY")
print("=" * 70)

for name, info in datasets.items():

    df = info["data"]

    possible = []

    for col in df.columns:

        col_clean = str(col).lower().strip()

        for keyword in MATCH_KEYWORDS:

            if keyword in col_clean:
                possible.append(col)
                break

    possible = list(dict.fromkeys(possible))

    print(f"\nDATASET: {name}")

    if possible:

        for col in possible:

            print(
                f"  {col}"
                f" | dtype={df[col].dtype}"
                f" | unique={df[col].nunique(dropna=True):,}"
            )

    else:

        print("  No obvious matching fields found.")

# ================================================================
# SPECIAL INSPECTION OF IMPORTANT INSTAGRAM FILES
# ================================================================

IMPORTANT_NAMES = [
    "Instagram - Posts",
    "Instagram - Posts2",
    "instagram_analytics",
    "instagram_reach",
    "Instagram - Posts.csv",
    "Instagram - Posts2.csv",
    "instagram_analytics.csv",
    "instagram_reach.csv"
]

print("\n" + "=" * 70)
print("IMPORTANT INSTAGRAM DATASET INSPECTION")
print("=" * 70)

for name, info in datasets.items():

    filename = info["path"].name.lower()

    if any(
        important.lower().replace(".csv", "") in
        filename.replace(".csv", "")
        for important in IMPORTANT_NAMES
    ):

        df = info["data"]

        print("\n" + "-" * 70)
        print(f"FILE: {info['path']}")
        print("-" * 70)

        print(f"Rows    : {len(df):,}")
        print(f"Columns : {len(df.columns)}")

        print("\nFULL COLUMN LIST:")

        for i, col in enumerate(df.columns, 1):
            print(f"{i:03d}. {col}")

        print("\nFIRST 5 ROWS:")

        print(
            df.head(5).to_string(
                max_cols=30,
                max_colwidth=50
            )
        )

# ================================================================
# NUMERIC COLUMN ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("NUMERIC ENGAGEMENT-RELATED FIELDS")
print("=" * 70)

numeric_candidates = []

for name, info in datasets.items():

    df = info["data"]

    for col in df.columns:

        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        col_clean = str(col).lower()

        if any(
            keyword in col_clean
            for keyword in TARGET_KEYWORDS
        ):

            numeric_candidates.append({
                "dataset": name,
                "column": col,
                "dtype": str(df[col].dtype),
                "count": int(df[col].count()),
                "missing": int(df[col].isna().sum()),
                "mean": float(df[col].mean()),
                "median": float(df[col].median()),
                "min": float(df[col].min()),
                "max": float(df[col].max())
            })

if numeric_candidates:

    numeric_df = pd.DataFrame(numeric_candidates)

    print(
        numeric_df.to_string(index=False)
    )

    numeric_file = RESULTS_DIR / "numeric_engagement_candidates.csv"

    numeric_df.to_csv(
        numeric_file,
        index=False
    )

    print("\nSaved:")
    print(numeric_file)

else:

    print("No numeric engagement candidates found.")

# ================================================================
# FINAL DIAGNOSTIC
# ================================================================

print("\n" + "=" * 70)
print("FINAL REAL-DATA TARGET DIAGNOSTIC")
print("=" * 70)

if target_candidates:

    print("\nPotential engagement/target fields FOUND:")
    print(f"Count: {len(target_candidates)}")

    for item in target_candidates:

        print(
            f"\nDataset : {item['dataset']}"
            f"\nColumn  : {item['column']}"
            f"\nType    : {item['dtype']}"
            f"\nMissing : {item['missing_pct']:.2f}%"
        )

    print(
        "\nNEXT STEP:"
        "\nSelect the correct engagement field and"
        "\nmatch it with the real image metadata."
    )

else:

    print("\nNO REAL ENGAGEMENT TARGET FOUND.")

    print(
        "\nNEXT STEP:"
        "\nInspect the available Instagram datasets manually"
        "\nand determine whether a real engagement variable exists."
    )

print("\n" + "=" * 70)
print("REAL TARGET DISCOVERY COMPLETED")
print("=" * 70)

REAL INSTAGRAM ENGAGEMENT TARGET DISCOVERY

Project root:
d:\newwwwwwww\AiBasedInstagramPrediction

Raw data directory:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw

CSV FILES FOUND
01. d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\Instagram - Posts.csv
02. d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\Instagram - Posts2.csv
03. d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_analytics.csv
04. d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_reach.csv
05. d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data\captions_csv.csv
06. d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data2\captions_csv2.csv

LOADING DATASETS

Instagram - Posts
  Rows    : 1,000
  Columns : 40
  Columns:
    - url
    - user_posted
    - description
    - hashtags
    - num_comments
    - date_posted
    - likes
    - photos
    - videos
    - location
    - latest_comments
    - post_id
    - discovery_input
    - has_handsh

In [2]:
# ================================================================
# REAL ENGAGEMENT TARGET - FOCUSED INSPECTION
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(r"d:\newwwwwwww\AiBasedInstagramPrediction")
RAW_DIR = PROJECT_ROOT / "datasets" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results" / "real_multimodal_evaluation"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "posts": RAW_DIR / "Instagram - Posts.csv",
    "posts2": RAW_DIR / "Instagram - Posts2.csv",
    "analytics": RAW_DIR / "instagram_analytics.csv",
    "reach": RAW_DIR / "instagram_reach.csv"
}

print("=" * 70)
print("REAL ENGAGEMENT TARGET - FOCUSED INSPECTION")
print("=" * 70)

# ---------------------------------------------------------------
# LOAD FILES
# ---------------------------------------------------------------

data = {}

for name, path in FILES.items():

    print(f"\n{'-' * 70}")
    print(f"{name.upper()}")
    print(f"FILE: {path}")
    print(f"{'-' * 70}")

    if not path.exists():
        print("STATUS: FILE NOT FOUND")
        continue

    try:
        df = pd.read_csv(path, low_memory=False)
        data[name] = df

        print(f"Rows    : {len(df):,}")
        print(f"Columns : {len(df.columns)}")

        print("\nColumns:")

        for i, col in enumerate(df.columns, 1):
            print(f"{i:02d}. {col}")

    except Exception as e:
        print(f"ERROR: {e}")

# ---------------------------------------------------------------
# SEARCH FOR ENGAGEMENT VARIABLES
# ---------------------------------------------------------------

keywords = [
    "like",
    "likes",
    "comment",
    "comments",
    "share",
    "shares",
    "save",
    "saves",
    "reach",
    "impression",
    "impressions",
    "engagement",
    "engagement_rate",
    "engagementrate",
    "interaction",
    "interactions",
    "view",
    "views"
]

print("\n" + "=" * 70)
print("POTENTIAL ENGAGEMENT VARIABLES")
print("=" * 70)

found = []

for dataset_name, df in data.items():

    print(f"\n[{dataset_name}]")

    dataset_found = []

    for col in df.columns:

        col_lower = str(col).lower().strip()

        if any(keyword in col_lower for keyword in keywords):

            dataset_found.append(col)

            series = df[col]

            info = {
                "dataset": dataset_name,
                "column": col,
                "dtype": str(series.dtype),
                "rows": len(series),
                "non_null": int(series.notna().sum()),
                "missing": int(series.isna().sum()),
                "unique": int(series.nunique(dropna=True))
            }

            found.append(info)

            print(
                f"\n  COLUMN: {col}"
                f"\n    Type       : {series.dtype}"
                f"\n    Non-null   : {series.notna().sum():,}"
                f"\n    Missing    : {series.isna().sum():,}"
                f"\n    Unique     : {series.nunique(dropna=True):,}"
            )

            if pd.api.types.is_numeric_dtype(series):

                print(f"    Min        : {series.min()}")
                print(f"    Median     : {series.median()}")
                print(f"    Mean       : {series.mean():.4f}")
                print(f"    Max        : {series.max()}")

            else:

                print("    Sample:")

                for value in series.dropna().astype(str).head(5):
                    print(f"      {value[:100]}")

    if not dataset_found:
        print("  No engagement-related columns found.")

# ---------------------------------------------------------------
# SHOW FIRST 5 ROWS OF IMPORTANT DATASETS
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET PREVIEW")
print("=" * 70)

for dataset_name, df in data.items():

    print(f"\n[{dataset_name}]")

    print(
        df.head(5).to_string(
            index=False,
            max_columns=20,
            max_colwidth=35
        )
    )

# ---------------------------------------------------------------
# FIND POSSIBLE POST / IMAGE IDENTIFIERS
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("IMAGE / POST MATCHING COLUMNS")
print("=" * 70)

match_keywords = [
    "image",
    "img",
    "file",
    "filename",
    "file_name",
    "post_id",
    "media_id",
    "media",
    "shortcode",
    "caption",
    "url",
    "id"
]

matching = []

for dataset_name, df in data.items():

    print(f"\n[{dataset_name}]")

    for col in df.columns:

        col_lower = str(col).lower()

        if any(keyword in col_lower for keyword in match_keywords):

            matching.append({
                "dataset": dataset_name,
                "column": col
            })

            print(f"  {col}")

# ---------------------------------------------------------------
# SAVE COMPACT SUMMARY
# ---------------------------------------------------------------

summary = pd.DataFrame(found)

summary_file = RESULTS_DIR / "real_engagement_field_summary.csv"

if not summary.empty:

    summary.to_csv(summary_file, index=False)

    print("\n" + "=" * 70)
    print("ENGAGEMENT FIELD SUMMARY")
    print("=" * 70)

    print(summary.to_string(index=False))

    print(f"\nSaved:")
    print(summary_file)

else:

    print("\n" + "=" * 70)
    print("NO ENGAGEMENT VARIABLES FOUND")
    print("=" * 70)

# ---------------------------------------------------------------
# FINAL STATUS
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL STATUS")
print("=" * 70)

if found:

    print(f"Potential engagement fields found: {len(found)}")

    print("\nWe now need to determine:")
    print("1. Which field represents real engagement")
    print("2. Whether it can be matched to the image records")
    print("3. How to create High / Medium / Low real labels")

else:

    print("No engagement fields were detected.")
    print("Real-data supervised evaluation cannot proceed yet.")

print("=" * 70)

REAL ENGAGEMENT TARGET - FOCUSED INSPECTION

----------------------------------------------------------------------
POSTS
FILE: d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\Instagram - Posts.csv
----------------------------------------------------------------------
Rows    : 1,000
Columns : 40

Columns:
01. url
02. user_posted
03. description
04. hashtags
05. num_comments
06. date_posted
07. likes
08. photos
09. videos
10. location
11. latest_comments
12. post_id
13. discovery_input
14. has_handshake
15. shortcode
16. content_type
17. pk
18. content_id
19. engagement_score_view
20. thumbnail
21. video_view_count
22. product_type
23. coauthor_producers
24. tagged_users
25. video_play_count
26. followers
27. posts_count
28. profile_image_link
29. is_verified
30. is_paid_partnership
31. partnership_details
32. user_posted_id
33. post_content
34. audio
35. profile_url
36. videos_duration
37. images
38. alt_text
39. photos_number
40. audio_url

-------------------------------------

TypeError: DataFrame.to_string() got an unexpected keyword argument 'max_columns'

In [3]:
# ---------------------------------------------------------------
# DATASET PREVIEW
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET PREVIEW")
print("=" * 70)

for dataset_name, df in data.items():

    print(f"\n[{dataset_name}]")

    preview = df.head(5).copy()

    # Limit long text values only for display
    for col in preview.columns:
        if preview[col].dtype == "object":
            preview[col] = preview[col].astype(str).str.slice(0, 80)

    print(preview.to_string(index=False))


DATASET PREVIEW

[posts]
                                       url     user_posted                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           description                                                                                                                                                              